In [16]:
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

In [17]:
# Get the MNIST dataset
# Set up the data preprocessing and loading:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])  # transform the data to torch tensor and normalize
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Now let's split the training set into a training and validation set
generator = torch.Generator().manual_seed(42)  # just a random generator
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [50000, 10000], generator=generator)

print('We loaded {} training, {} validation, and {} testing samples'.format(len(train_dataset), len(val_dataset), len(test_dataset)))

# Set up the data loaders (they are iterable objects that return the data in batches)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

We loaded 50000 training, 10000 validation, and 10000 testing samples


In [55]:
data, labels = next(iter(train_loader))
print(labels)

tensor([0, 3, 8, 2, 5, 5, 6, 4, 9, 6, 5, 3, 3, 6, 2, 5, 1, 3, 5, 9, 8, 7, 0, 0,
        8, 6, 6, 4, 4, 5, 2, 5, 4, 5, 9, 7, 4, 8, 0, 8, 3, 8, 6, 3, 9, 9, 0, 4,
        8, 1, 4, 9, 2, 0, 0, 6, 8, 7, 1, 3, 3, 7, 4, 8])


In [40]:
data.shape

torch.Size([64, 1, 28, 28])

In [ ]:
# MINE statistics network for first convolutional layer
# I(X, T)
class MINEFirstIn(nn.Module):
    def __init__(self):
        super(MINEFirstIn, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  
        self.reluconv1 = nn.ReLU() 
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.conv2 = nn.Conv2d(64, 32, kernel_size=3, padding=1) 
        self.reluconv2 = nn.ReLU()
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)  
        self.fc1 = nn.Linear(7*7*32, 16)  
        self.relufc1 = nn.ReLU() 
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x, t):
        x = self.reluconv1(self.conv1(x))
        h = torch.cat([x, t], dim = 1)
        h = self.maxpool1(h)
        h = self.reluconv2(self.conv2(h))
        h = self.maxpool2(h)
        h = h.view(-1, 7*7*32)
        h = self.relufc1(self.fc1(h))
        h = self.fc2(h)
        return h

# I(T, Y)
class MINEFirstOut(nn.Module):
    def __init__(self):
        super(MINEFirstOut, self).__init__()  
        self.reluconv1 = nn.ReLU() 
        self.maxpool1 = nn.MaxPool2d(kernel_size=4, stride=4) 
        self.fc1 = nn.Linear(7*7*32 + 10, 16)  
        self.relufc1 = nn.ReLU() 
        self.fc2 = nn.Linear(16, 1)

    def forward(self, t, y):
        t = self.reluconv1(t)
        t = self.maxpool1(t)
        t = t.view(-1, 7*7*32)
        h = torch.cat([y, t], dim = 1)
        h = self.relufc1(self.fc1(h))
        h = self.fc2(h)
        return h

In [35]:
def smoothed_one_hot(label: int, epsilon: float):
    z = torch.zeros(10)
    z[label] = 1
    return (1-epsilon)*z + epsilon/10

In [ ]:
# REVISAR EMA (Exponential Moving Average)
def donsker_varadhan_loss(joint, marginal):
    joint_mean = torch.mean(joint)
    marginal_mean = torch.mean(torch.exp(marginal))

    alpha = 0.01
    ema_start = torch.tensor(1.0)

    with torch.no_grad():
        ema_marginal = (1 - alpha) * ema_start + alpha * marginal_mean.detach()

    estimate = joint_mean - torch.log(ema_marginal)
    loss = -estimate
    return loss, estimate.detach()